# WGBS/RRBS Processing with Bismark

**Tier 3 — Applied Bioinformatics | Module 32 · Notebook 1**

*Prerequisites: Module 01 (NGS Fundamentals), Module 24 (ChIP-seq & Epigenomics)*

---

**By the end of this notebook you will be able to:**
1. Describe 5-methylcytosine (5mC) biology and CpG island methylation patterns
2. Explain bisulfite conversion chemistry and its effect on sequencing reads
3. Run the Bismark pipeline: bisulfite index → trimming → alignment → methylation extraction
4. Distinguish WGBS, RRBS, and EPIC array approaches
5. Compute CpG methylation levels and visualize genome-wide methylation landscapes

**Key resources:**
- [Bismark documentation](https://felixkrueger.github.io/Bismark/)
- [methylKit vignette](https://bioconductor.org/packages/release/bioc/vignettes/methylKit/)
- [Encode WGBS pipeline](https://www.encodeproject.org/wgbs/)

---[< Previous: Single-Cell Batch Correction and Dataset Integr...](../31_Single_Cell_Multi_Omics/03_sc_integration.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Differentially Methylated Regions (DMRs) >](02_dmr_analysis.ipynb)---

## 1. DNA Methylation Biology

**5-methylcytosine (5mC)** is the most studied epigenetic mark in mammals. A methyl group is added to the 5-carbon position of cytosine by DNA methyltransferases (DNMTs): DNMT3A and DNMT3B establish de novo methylation, while DNMT1 maintains methylation through replication. The reverse reaction is catalysed by TET enzymes (TET1/2/3), which oxidize 5mC to 5-hydroxymethylcytosine (5hmC) and beyond, ultimately leading to passive or active demethylation.

### Genomic context of cytosine methylation

In mammals, methylation occurs almost exclusively at **CpG dinucleotides**. Roughly 70–80% of all CpGs in the human genome are methylated. The exceptions are:

| Feature | Methylation state | Function |
|---|---|---|
| CpG islands (CGIs) | Mostly **unmethylated** | Protect gene promoters from silencing |
| CGI shores (±2 kb) | Variable, tissue-specific | Major site of differential methylation |
| Gene bodies | Moderately methylated | Associated with active transcription |
| Repeats / transposons | Heavily methylated | Silences parasitic elements |
| Imprinted regions | Hemi-methylated (parent-of-origin) | Monoallelic gene expression |

**CpG islands** are defined as regions ≥200 bp with CpG observed/expected ratio > 0.6 and GC content > 50%. Despite representing only ~1% of the genome, they overlap ~70% of gene promoters. Their unmethylated state allows transcription factor binding.

### Methylation in development and disease

- During early embryogenesis, methylation is globally erased and re-established in a lineage-specific manner — explaining why different tissues have distinct methylation profiles.
- In cancer, CpG island promoter **hypermethylation** silences tumor suppressor genes (e.g., *BRCA1*, *MLH1*, *CDKN2A*), while global hypomethylation destabilizes the genome.
- Aging is associated with a gradual drift in methylation — the basis for epigenetic clocks (Notebook 3).

### Why measure methylation at single-CpG resolution?

Array-based methods (Illumina 450K/EPIC) measure 450,000–850,000 CpGs but miss most of the genome. **Whole-Genome Bisulfite Sequencing (WGBS)** provides base-resolution methylation across all ~28 million CpGs in the human genome, at the cost of sequencing depth (~30× per strand).

## 2. Bisulfite Sequencing Chemistry

The key insight behind bisulfite sequencing is a simple chemical reaction:

> **Unmethylated cytosine + sodium bisulfite → uracil → reads as T**  
> **Methylated cytosine + sodium bisulfite → protected → reads as C**

This allows methylation status to be inferred at single-base resolution by comparing the bisulfite-treated sequence to the reference genome: a C in the read means the site was methylated; a T means it was unmethylated.

### Step-by-step chemistry

1. **Denaturation**: DNA is denatured to single strands (required for bisulfite access).
2. **Sulfonation**: Bisulfite adds to the C5–C6 double bond of cytosine, forming cytosine-sulfonate.
3. **Deamination**: The amino group is removed, yielding uracil-sulfonate.
4. **Desulfonation**: Under alkaline conditions, the sulfonate is removed, leaving uracil.
5. **PCR amplification**: Uracil is copied as thymine.

Methylated cytosines resist sulfonation because the methyl group at C5 sterically blocks the reaction.

### Bisulfite conversion efficiency

Incomplete conversion (unconverted unmethylated C) is a critical quality metric. It is estimated using:
- **Lambda spike-in**: Bacteriophage lambda DNA (known to be unmethylated) is spiked in and used to compute non-conversion rate.
- **CHH/CHG sites**: In mammalian cells, non-CpG methylation is rare, so CHH/CHG C→T conversion rate reflects bisulfite efficiency.
- **Target**: > 99.5% conversion efficiency.

### WGBS vs RRBS vs EPIC array

| Method | Coverage | Cost | Best for |
|---|---|---|---|
| WGBS | All ~28M CpGs | High (>$500/sample) | Comprehensive studies |
| RRBS | ~5M CpGs (CpG-enriched) | Medium | Cost-effective targeted |
| EPIC array | 850K CpGs | Low (~$200) | Large cohorts |

**RRBS** uses MspI restriction enzyme (cuts at CCGG) to enrich CpG-dense regions before bisulfite treatment, reducing sequencing cost ~6× while covering most CpG islands.

### Effect on alignment

Bisulfite treatment creates four distinct strands (OT, CTOT, OB, CTOB), each with a different C→T conversion pattern. Standard aligners cannot handle this complexity — specialized aligners like **Bismark** are required.

## 3. Bismark Alignment Pipeline

Bismark is the standard tool for aligning bisulfite-converted reads. It works by:

1. **Index preparation**: Converting all cytosines in the reference genome to thymine (CT conversion) and also converting all guanines on the reverse complement to adenine (GA conversion), then building Bowtie2 indices for both strands.
2. **Read alignment**: For each read, Bismark performs a CT conversion and aligns against the CT-converted genome; it also performs a GA conversion and aligns against the GA-converted genome. The best alignment wins.
3. **Methylation extraction**: Comparing aligned reads to the original (unconverted) reference, calling each CpG site as methylated (C in read) or unmethylated (T in read).

### Step 1: Build the bisulfite genome index

```bash
# Prepare genome index (only needs to be done once per genome)
bismark_genome_preparation /path/to/genome/hg38/

# This creates CT_conversion/ and GA_conversion/ subdirectories
# with Bowtie2 indices
```

### Step 2: Quality trimming

Bisulfite-treated libraries require aggressive adapter trimming (Trim Galore wraps cutadapt with bisulfite-specific settings):

```bash
trim_galore --paired --fastqc \
    sample_R1.fastq.gz sample_R2.fastq.gz
```

### Step 3: Alignment

```bash
bismark --genome /path/to/genome/hg38/ \
    -1 sample_R1_val_1.fq.gz \
    -2 sample_R2_val_2.fq.gz \
    --output_dir bismark_output/
```

Key output: `sample_R1_val_1_bismark_bt2_pe.bam`

### Step 4: Deduplication

PCR duplicates are problematic in bisulfite sequencing because two reads mapping to the same position could represent the same molecule (since bisulfite treatment reduces sequence complexity). Deduplication is essential for WGBS (less critical for RRBS, where many reads genuinely overlap).

```bash
deduplicate_bismark -p sample_R1_val_1_bismark_bt2_pe.bam
```

### Step 5: Methylation extraction

```bash
bismark_methylation_extractor \
    --paired-end \
    --CpG \
    --CHG \
    --CHH \
    --comprehensive \
    --cytosine_report \
    --genome_folder /path/to/genome/hg38/ \
    sample_R1_val_1_bismark_bt2_pe.deduplicated.bam
```

**Output file formats:**
- `*.bismark.cov.gz`: One line per CpG — chromosome, start, end, % methylated, methylated count, unmethylated count
- `*.CpG_report.txt.gz`: Full genome coverage, including sites with zero coverage
- `CpX_context_*`: Separate files for CpG, CHG, CHH contexts

### Methylation contexts

| Context | Definition | Mammalian prevalence | Notes |
|---|---|---|---|
| CpG | CG dinucleotide | ~70% methylated | Most studied; heritable |
| CHG | Where H = A, T, or C | <1% in somatic cells | Common in plants |
| CHH | H = A, T, or C on both strands | <1% in somatic cells | Non-CpG methylation in neurons |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

np.random.seed(42)

# Simulate a Bismark coverage file: columns are chromosome, start, end,
# pct_methylated, count_methylated, count_unmethylated
# We model three genomic contexts with realistic methylation distributions:
#   - CpG islands (promoters): low methylation (~10%)
#   - Gene bodies: moderate methylation (~60%)
#   - Intergenic / repeat regions: high methylation (~85%)

n_islands   = 3000   # CpG island CpGs
n_genebody  = 8000   # gene body CpGs
n_intergenic = 9000  # repeat / intergenic CpGs

def simulate_cpgs(n, mu_beta, sigma_beta, context_label, coverage_mean=20):
    """Simulate CpG sites with beta-distributed methylation and negative-binomial coverage."""
    # Beta parameters from mean/variance
    alpha = mu_beta * ((mu_beta * (1 - mu_beta)) / sigma_beta**2 - 1)
    beta_param = (1 - mu_beta) * ((mu_beta * (1 - mu_beta)) / sigma_beta**2 - 1)
    alpha = max(alpha, 0.5)
    beta_param = max(beta_param, 0.5)
    
    betas = np.random.beta(alpha, beta_param, n)
    betas = np.clip(betas, 0, 1)
    
    # Coverage: negative binomial (overdispersed counts typical of sequencing)
    coverage = np.random.negative_binomial(n=5, p=5/(5+coverage_mean), size=n)
    coverage = np.maximum(coverage, 1)   # at least 1 read
    
    count_M = np.round(betas * coverage).astype(int)
    count_U = coverage - count_M
    
    return pd.DataFrame({
        'beta': betas,
        'coverage': coverage,
        'count_M': count_M,
        'count_U': np.maximum(count_U, 0),
        'context': context_label
    })

df_island    = simulate_cpgs(n_islands,    mu_beta=0.08, sigma_beta=0.08, context_label='CpG Island')
df_genebody  = simulate_cpgs(n_genebody,   mu_beta=0.60, sigma_beta=0.18, context_label='Gene Body')
df_intergenic = simulate_cpgs(n_intergenic, mu_beta=0.84, sigma_beta=0.12, context_label='Intergenic/Repeat')

cpg_data = pd.concat([df_island, df_genebody, df_intergenic], ignore_index=True)

# Recompute beta from count data (as Bismark does)
cpg_data['beta_from_counts'] = cpg_data['count_M'] / (cpg_data['count_M'] + cpg_data['count_U'])

print(f"Total CpG sites simulated: {len(cpg_data):,}")
print(f"\nPer-context summary:")
print(cpg_data.groupby('context')['beta_from_counts'].describe().round(3))
print(f"\nFormula:  beta = count_M / (count_M + count_U)")

## 4. The Beta Value: Quantifying Methylation

The **beta value** (β) is the fraction of methylated reads at a CpG site:

$$\beta = \frac{M}{M + U}$$

where M = count of methylated reads, U = count of unmethylated reads. Beta values range from 0 (completely unmethylated) to 1 (fully methylated).

An alternative representation is the **M-value** (logit transform), preferred for statistical testing because it is more homoscedastic (constant variance across the [0,1] range):

$$M = \log_2\!\left(\frac{\beta + \varepsilon}{1 - \beta + \varepsilon}\right)$$

**Coverage filtering** is crucial: low-coverage CpGs produce unreliable beta values due to sampling noise. A common threshold is ≥ 10× coverage (some strict analyses require ≥ 20×).

Below we visualize the genome-wide beta distribution and show why context matters.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = {'CpG Island': '#2196F3', 'Gene Body': '#4CAF50', 'Intergenic/Repeat': '#FF5722'}

# Panel 1: Beta value distributions per context
for ctx, grp in cpg_data.groupby('context'):
    axes[0].hist(grp['beta_from_counts'], bins=50, alpha=0.65,
                 color=colors[ctx], label=ctx, density=True)
axes[0].set_xlabel('Beta value (methylation fraction)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Genome-wide Methylation Distribution\nby Genomic Context', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1)

# Panel 2: Coverage distribution
cov_clip = np.clip(cpg_data['coverage'], 0, 60)
axes[1].hist(cov_clip, bins=40, color='#9C27B0', alpha=0.75, edgecolor='white')
axes[1].axvline(10, color='red', linestyle='--', linewidth=1.5, label='Min coverage (10×)')
axes[1].set_xlabel('Coverage (reads per CpG)', fontsize=11)
axes[1].set_ylabel('Number of CpG sites', fontsize=11)
axes[1].set_title('Read Coverage Distribution\n(negative binomial model)', fontsize=11)
axes[1].legend(fontsize=9)

# Panel 3: Beta vs M-value scatter
eps = 0.01
sample = cpg_data.sample(2000, random_state=1)
m_values = np.log2((sample['beta_from_counts'] + eps) / (1 - sample['beta_from_counts'] + eps))
scatter_colors = [colors[c] for c in sample['context']]
axes[2].scatter(sample['beta_from_counts'], m_values, c=scatter_colors, alpha=0.3, s=5)
axes[2].set_xlabel('Beta value', fontsize=11)
axes[2].set_ylabel('M-value (logit)', fontsize=11)
axes[2].set_title('Beta vs M-value Transformation\n(logit scale more homoscedastic)', fontsize=11)
# Add legend patches
patches = [mpatches.Patch(color=v, label=k) for k, v in colors.items()]
axes[2].legend(handles=patches, fontsize=8)

plt.tight_layout()
plt.savefig('wgbs_methylation_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 5. Coverage Filtering and Quality Metrics

Before any downstream analysis, CpG sites must pass quality filters. Low-coverage sites have high binomial sampling variance — a site covered by only 3 reads and observed as methylated could be 33%, 67%, or 100% — all plausible by chance. We apply:

1. **Minimum coverage filter**: discard CpGs with < 10 reads total.
2. **Maximum coverage filter**: very high coverage (> 99th percentile) may indicate PCR duplicates that survived deduplication — these sites can have artificially low variance.
3. **Non-conversion check**: sites where the CHH/CHG context has high apparent methylation indicate poor bisulfite conversion in that region.

The **bismark2summary** command produces an HTML QC report after each alignment including:
- Alignment rate (expected: 60–80% for WGBS; low alignment can indicate bisulfite conversion failure or contamination)
- Unique vs multi-mapping reads
- Strand-specific alignment statistics
- Non-conversion rate (CHH methylation as proxy)

In [ ]:
# Apply coverage filtering and report how many CpGs pass each threshold
min_cov = 10
max_cov = np.percentile(cpg_data['coverage'], 99)

mask_pass = (cpg_data['coverage'] >= min_cov) & (cpg_data['coverage'] <= max_cov)
cpg_filtered = cpg_data[mask_pass].copy()

print("=== Coverage Filtering Summary ===")
print(f"Total CpG sites:         {len(cpg_data):>8,}")
print(f"Pass min coverage (≥10): {mask_pass.sum():>8,}  ({mask_pass.mean()*100:.1f}%)")
print(f"Max coverage cutoff:     {max_cov:>8.0f}x  (99th percentile)")
print(f"Sites retained:          {len(cpg_filtered):>8,}")

# Show how coverage affects precision: binomial 95% CI width
coverages = np.array([5, 10, 20, 30, 50])
ci_widths = []
for cov in coverages:
    # For a true beta of 0.5 (worst case), 95% CI width under binomial
    lo = stats.binom.ppf(0.025, cov, 0.5) / cov
    hi = stats.binom.ppf(0.975, cov, 0.5) / cov
    ci_widths.append(hi - lo)

print("\n=== Binomial 95% CI Width at beta=0.5 (worst case) ===")
for cov, w in zip(coverages, ci_widths):
    print(f"  Coverage {cov:>3}x : CI width = {w:.3f} ({w*100:.1f} percentage points)")

## 6. Visualizing a Methylation Landscape

A key deliverable in any WGBS study is a **genomic methylation track** — showing methylation level across a chromosomal region. Here we simulate a small genomic segment that spans:
- An unmethylated CpG island (promoter of a housekeeping gene)
- The gene body with moderate methylation
- A downstream intergenic region with high methylation

This bimodal pattern — low at active promoters, high elsewhere — is the canonical methylation landscape of somatic cells.

In [ ]:
np.random.seed(7)

# Simulate a 50 kb genomic region (positions in bp)
# Annotated features:
#   0 - 2000:   5' intergenic (high methylation)
#   2000 - 3500: CpG island / promoter (low methylation)
#   3500 - 3700: CGI shore (intermediate)
#   3700 - 25000: gene body (moderate methylation)
#   25000 - 50000: intergenic (high methylation)

def make_segment(start, end, mu, sigma, spacing=50):
    positions = np.arange(start, end, spacing)
    alpha_b = mu * ((mu*(1-mu))/sigma**2 - 1)
    beta_b  = (1-mu) * ((mu*(1-mu))/sigma**2 - 1)
    betas   = np.random.beta(max(alpha_b,0.5), max(beta_b,0.5), len(positions))
    return positions, np.clip(betas, 0, 1)

segments = [
    (0,    2000,  0.85, 0.08),   # 5' intergenic
    (2000, 3500,  0.05, 0.04),   # CpG island
    (3500, 3700,  0.35, 0.12),   # CGI shore
    (3700, 25000, 0.60, 0.15),   # gene body
    (25000,50000, 0.85, 0.08),   # intergenic
]

all_pos  = []
all_beta = []
for s in segments:
    p, b = make_segment(*s)
    all_pos.extend(p)
    all_beta.extend(b)

all_pos  = np.array(all_pos)
all_beta = np.array(all_beta)

fig, ax = plt.subplots(figsize=(14, 4))

ax.scatter(all_pos / 1000, all_beta, s=8, c=all_beta,
           cmap='RdYlGn_r', vmin=0, vmax=1, alpha=0.7)

# Shaded feature regions
feature_regions = [
    (2.0, 3.5, '#2196F3', 'CpG Island'),
    (3.5, 3.7, '#9C27B0', 'CGI Shore'),
    (3.7, 25.0,'#4CAF50', 'Gene Body'),
]
for start_kb, end_kb, col, label in feature_regions:
    ax.axvspan(start_kb, end_kb, alpha=0.12, color=col, label=label)

ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.set_xlabel('Genomic position (kb)', fontsize=11)
ax.set_ylabel('Methylation (beta value)', fontsize=11)
ax.set_title('Simulated WGBS Methylation Track\n(CpG island promoter → gene body → intergenic)', fontsize=11)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='lower right', fontsize=9)

sm = plt.cm.ScalarMappable(cmap='RdYlGn_r', norm=plt.Normalize(0, 1))
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Beta value', shrink=0.8)

plt.tight_layout()
plt.savefig('wgbs_landscape.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. EPIC Array Comparison

When WGBS sequencing is not feasible (cost, sample quantity, cohort size), the **Illumina EPIC array** (850K) is the standard alternative. It interrogates ~850,000 pre-selected CpG sites via bisulfite-converted hybridization, producing beta values directly from red/green fluorescence intensities.

### Key differences from WGBS

| Feature | WGBS | EPIC Array |
|---|---|---|
| CpG coverage | ~28 million | ~850,000 |
| Coverage of CGIs | ~95% | ~55% |
| Non-CpG methylation | Yes | No |
| Cost per sample | ~$500–1500 | ~$150–300 |
| Input DNA | 0.5–5 µg | 250 ng–1 µg |
| Typical use | Pilot/discovery | Large cohorts |

### minfi preprocessing (R)

```r
library(minfi)
# Load IDAT files
RGSet <- read.metharray.exp("idat_directory/")
# Normalize: Noob (background subtraction) + BMIQ (type I/II probe bias)
MSet <- preprocessNoob(RGSet)
beta  <- getBeta(MSet)   # matrix: CpGs × samples, values 0–1
```

### Where arrays fall short

Arrays miss most of the genome. CGI **shores** (±2 kb from CGI) — which show the most tissue-specific differential methylation — are only partially covered. Enhancer methylation and repeat element methylation are largely absent. For novel DMR discovery or cancer epigenome analysis, WGBS is necessary.

## 8. Summary and Key Takeaways

This notebook covered the complete WGBS processing pipeline:

1. **Biology**: 5mC is deposited by DNMT3A/3B, maintained by DNMT1, and erased by TET enzymes. Methylation is context-dependent — CpG islands at active promoters are unmethylated; intergenic and repeat regions are heavily methylated.

2. **Chemistry**: Bisulfite converts unmethylated C → T; methylated C is protected. Non-conversion rate (ideally < 0.5%) is a critical QC metric. WGBS provides complete coverage; RRBS and arrays are cost-effective alternatives.

3. **Bismark pipeline**:
   - `bismark_genome_preparation` → build CT/GA-converted indices
   - `trim_galore` → adapter/quality trimming
   - `bismark` → bisulfite-aware alignment (~60–80% mapping rate)
   - `deduplicate_bismark` → remove PCR duplicates
   - `bismark_methylation_extractor` → CpG/CHG/CHH context files

4. **Beta value**: β = M/(M+U), range [0,1]. Apply coverage filters (≥10×) before analysis. M-values (logit-transformed beta) are better for linear models.

5. **EPIC arrays**: cheaper, lower CpG coverage, adequate for large cohort studies. WGBS is necessary for comprehensive or cancer epigenome studies.

**Next**: Notebook 2 covers how to detect **differentially methylated regions (DMRs)** between conditions using statistical approaches including BSmooth and DSS.

---[< Previous: Single-Cell Batch Correction and Dataset Integr...](../31_Single_Cell_Multi_Omics/03_sc_integration.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Differentially Methylated Regions (DMRs) >](02_dmr_analysis.ipynb)---